# KLRfome synthetic methods laboratory

This notebook runs or loads controlled M0–M3 experiments. Synthetic labels represent known distribution classes; scores remain relative rankings rather than occurrence probabilities. Start with the smoke configuration, then switch to the core configuration for research runs.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPOSITORY = Path.cwd()
if not (REPOSITORY / 'benchmarks').exists():
    REPOSITORY = REPOSITORY.parent
CONFIGURATION = REPOSITORY / 'benchmarks/synthetic_lab_smoke_config.json'
OUTPUT = REPOSITORY / 'benchmark_data/synthetic_lab_notebook_results.json'
RUN_EXPERIMENT = False
sns.set_theme(style='whitegrid')

## Run or load results

Set `RUN_EXPERIMENT=True` to execute the selected configuration. Full research output remains outside version control under `benchmark_data/`.

In [ ]:
if RUN_EXPERIMENT:
    subprocess.run(
        [
            sys.executable,
            str(REPOSITORY / 'benchmarks/run_synthetic_methods_lab.py'),
            '--config', str(CONFIGURATION),
            '--output', str(OUTPUT),
        ],
        cwd=REPOSITORY,
        check=True,
    )

if not OUTPUT.exists():
    raise FileNotFoundError('Run the experiment or point OUTPUT to an existing result file.')
result = json.loads(OUTPUT.read_text())
print(result['configuration_sha256'])
print(f"{len(result['cases'])} cases loaded")

In [ ]:
records = []
for case in result['cases']:
    scenario = case['scenario']
    for row in case['fold_results']:
        records.append({
            'case_id': case['case_id'],
            'scenario': scenario['scenario'],
            'effect_size': scenario['effect_size'],
            'bag_size': scenario['bag_size'],
            'spatial_range': scenario['spatial_range'],
            'seed': scenario['seed'],
            **row,
        })
folds = pd.DataFrame(records)
folds.head()

## Paired performance summaries

In [ ]:
summary = (
    folds.groupby(['scenario', 'effect_size', 'bag_size', 'method'], dropna=False)
    .agg(
        mean_auc=('auc', 'mean'),
        mean_pr_auc=('pr_auc', 'mean'),
        mean_boyce=('boyce', 'mean'),
        mean_lift=('top_5_percent_lift', 'mean'),
        median_fit_seconds=('fit_seconds', 'median'),
        median_predict_seconds=('predict_seconds', 'median'),
    )
    .reset_index()
)
summary

In [ ]:
g = sns.catplot(
    data=folds, x='method', y='auc', col='scenario', hue='method',
    kind='box', sharey=True, col_wrap=3, height=3.2, legend=False,
)
g.set_xticklabels(rotation=45, ha='right')
g.set_axis_labels('', 'Fold ROC AUC')
plt.show()

## Bag size, dependence, and generalization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.lineplot(data=folds, x='bag_size', y='auc', hue='method', marker='o', ax=axes[0])
axes[0].set_title('Bag-size sensitivity')
sns.boxplot(data=folds, x='method', y='auc_generalization_gap', ax=axes[1])
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_title('Train–test AUC gap')
plt.tight_layout()
plt.show()

## M1 approximation diagnostics

In [ ]:
m1_diagnostics = []
for case in result['cases']:
    for row in case['fold_results']:
        approximation = row.get('diagnostics', {}).get('kernel_approximation')
        agreement = row.get('diagnostics', {}).get('score_agreement_with_M0')
        if approximation:
            m1_diagnostics.append({
                'case_id': case['case_id'], 'method': row['method'],
                **approximation, **{f'score_{k}': v for k, v in agreement.items()},
            })
pd.DataFrame(m1_diagnostics)

## Invariance and intentional reweighting

In [ ]:
invariance_rows = []
for case in result['cases']:
    for method, values in (case.get('invariance') or {}).items():
        invariance_rows.append({'case_id': case['case_id'], 'method': method, **values})
pd.DataFrame(invariance_rows)

## Interpretation gate

Use controlled failures to select the next extension: poor M1 fidelity motivates improved random features; small-bag instability motivates shrinkage; sparse-signal failure motivates ARD or grouped kernels; nonlinear gains support M2; and shape-sensitive gains support M3 or hybrid kernels. Do not promote or remove a method from synthetic results alone.